In [26]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from scipy.stats import spearmanr, pearsonr
import torch

from models_transformer import SingleOutTransformerNet

In [14]:
shap_arr = np.load("Results/shap_values/shap_values.npz")
shap_f = shap_arr["shap_vals"]

In [20]:
rollout_dict = np.load("Results/rollout_local/rollout.npz")
R_test = rollout_dict['rollout']

In [22]:
test_data = np.load("data_processed/test_data_scaled.npz")
X_test = test_data["x"]    

(1594, 72)

In [31]:
IN_DIM = X_test.shape[1]
DEVICE = torch.device("cpu")

EMB_DIM = 64
NHEAD = 4
NUM_LAYERS = 3
# NUM_LAYERS = 5
FF_DIM = 128
DROPOUT = 0.1

model = SingleOutTransformerNet(IN_DIM, emb_dim=EMB_DIM, nhead=NHEAD, 
                                num_layers=NUM_LAYERS, ff_dim=FF_DIM, 
                                dropout=DROPOUT).to(DEVICE)

state_dict = torch.load(f"trained_models/TR_model_{NUM_LAYERS}layers.pt")
model.load_state_dict(state_dict)
model.eval()

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

with torch.no_grad():
    y_pred = model(X_test_tensor)

f_pred_test = y_pred.detach().cpu().numpy().squeeze()

(1594,)


In [32]:
# Inputs you already have:
# X_test:        shape (N, d)
# R_test:        shape (N, d, d), where R_test[n] = R(x_n)
# f_pred_test:  shape (N,), transformer predictions f(x_n)
# shap_f:       shape (N, d), SHAP values of the transformer f

def rollout_features(X_test, R_test):
    """
    Computes z_n = R(x_n) x_n for every test sample.
    """
    return np.einsum("nij,nj->ni", R_test, X_test)


# --------------------------------------------------
# 1. Build rollout surrogate g_roll(x) = beta^T R(x)x
# --------------------------------------------------

Z_test = rollout_features(X_test, R_test)  # shape (N, d)

# Fit beta_roll on the test set only if this is just a diagnostic.
# Better: fit beta_roll on training data and evaluate on test data.
ridge = Ridge(alpha=1e-6, fit_intercept=True)
ridge.fit(Z_test, f_pred_test)

beta_roll = ridge.coef_          # shape (d,)
b_roll = ridge.intercept_

g_pred_test = ridge.predict(Z_test)
e_pred_test = f_pred_test - g_pred_test


# --------------------------------------------------
# 2. Approximate SHAP values of the rollout surrogate
# --------------------------------------------------
# For each sample:
# g_roll(x_n) = beta^T R_n x_n
# If R_n is frozen locally, this is linear in x with coefficient
# a_n = R_n^T beta.

x_baseline = X_test.mean(axis=0)  # or use the same background mean used for SHAP

local_coeffs = np.einsum("nij,j->ni", np.transpose(R_test, (0, 2, 1)), beta_roll)
# equivalently: local_coeffs[n] = R_test[n].T @ beta_roll

shap_g = local_coeffs * (X_test - x_baseline)  # shape (N, d)


# --------------------------------------------------
# 3. Residual SHAP values by linearity
# --------------------------------------------------

shap_e = shap_f - shap_g

I_f = np.mean(np.abs(shap_f), axis=0)
I_g = np.mean(np.abs(shap_g), axis=0)
I_e = np.mean(np.abs(shap_e), axis=0)

rho_e_global = np.sum(I_e) / (np.sum(I_f) + 1e-12)
rho_e_feature = I_e / (I_f + 1e-12)


# --------------------------------------------------
# 4. Diagnostics
# --------------------------------------------------

print("Rollout surrogate R^2 on test:")
ss_res = np.sum((f_pred_test - g_pred_test) ** 2)
ss_tot = np.sum((f_pred_test - np.mean(f_pred_test)) ** 2)
print(1 - ss_res / ss_tot)

print("\nGlobal residual attribution ratio:")
print(rho_e_global)

print("\nMean feature-wise residual ratio:")
print(np.mean(rho_e_feature))

print("\nMedian feature-wise residual ratio:")
print(np.median(rho_e_feature))

print("\nCorrelation between full SHAP importance and rollout-surrogate SHAP importance:")
print("Pearson:", pearsonr(I_f, I_g)[0])
print("Spearman:", spearmanr(I_f, I_g)[0])


# --------------------------------------------------
# 5. Optional: create a feature-level summary table
# --------------------------------------------------

def residual_diagnostic_table(feature_names=None):
    d = X_test.shape[1]
    if feature_names is None:
        feature_names = [f"feature_{j}" for j in range(d)]

    table = []
    for j in range(d):
        table.append({
            "feature": feature_names[j],
            "I_f": I_f[j],
            "I_g_roll": I_g[j],
            "I_e_residual": I_e[j],
            "residual_ratio": rho_e_feature[j],
        })

    table = sorted(table, key=lambda row: row["I_f"], reverse=True)
    return table

diagnostic_table = residual_diagnostic_table()

for row in diagnostic_table[:15]:
    print(row)

Rollout surrogate R^2 on test:
0.8381978124380112

Global residual attribution ratio:
0.7195819073038932

Mean feature-wise residual ratio:
0.9876274412659514

Median feature-wise residual ratio:
0.7447453083500069

Correlation between full SHAP importance and rollout-surrogate SHAP importance:
Pearson: 0.6715731571877062
Spearman: 0.6494951443822755
{'feature': 'feature_9', 'I_f': 0.15403487493789741, 'I_g_roll': 0.110718206, 'I_e_residual': 0.09286613199426202, 'residual_ratio': 0.602890300207016}
{'feature': 'feature_43', 'I_f': 0.09726861945735442, 'I_g_roll': 0.07364993, 'I_e_residual': 0.050772870625191924, 'residual_ratio': 0.5219861339445692}
{'feature': 'feature_26', 'I_f': 0.09709164362561361, 'I_g_roll': 0.009191966, 'I_e_residual': 0.09153199968787563, 'residual_ratio': 0.9427381829056395}
{'feature': 'feature_32', 'I_f': 0.09520157617974552, 'I_g_roll': 0.052773155, 'I_e_residual': 0.051403984776865114, 'residual_ratio': 0.5399488836116723}
{'feature': 'feature_0', 'I_f': 